# Phase 2 – Technical Application: Reproducing the Paper's Pipeline and Fixing Its Flaws
**Paper reviewed:** *Performance Evaluation of Machine Learning and Deep Learning Models for Credit Risk Prediction*
**Data:** German Credit (UCI Statlog, n = 1,000) and Taiwan Credit (UCI, n = 30,000), analysed **separately**

## What this notebook does
1. **Reproduces the paper's pipeline as described** (`paper_style`): unspecified encoding, no scaling, SMOTE-ENN on the encoded data.
2. **Implements corrected pipelines** that fix each flaw found in the Phase 1 review.
3. **Tests whether the fixes matter** with repeated stratified CV and a corrected resampled t-test.
4. **Compares the best corrected model with the authors' reported results** using a verdict rule fixed in advance.

## Flaws found in Phase 1 → fix implemented here
| Flaw in the paper | Fix in this notebook |
|---|---|
| Encoding not stated; SMOTE interpolates over encoded categories (invalid fractional codes) | One-hot encoding in the model, **SMOTE-NC** for resampling mixed data |
| Scaling not stated (distance-based SMOTE/ENN, MLP) | In-fold standardisation after a signed-log transform of skewed amounts |
| Heavily skewed monetary features, no transformation | `SkewLog` transformer fitted **inside each training fold** |
| Missing values, duplicates, undocumented codes not addressed | Documented cleaning step with counts printed (35 Taiwan duplicates, EDUCATION 0/5/6, MARRIAGE 0, PAY_0 naming) |
| Accuracy/F1 only; no cost-sensitive or threshold-free metrics | MCC, balanced accuracy, ROC-AUC, **PR-AUC**, Brier score, and the German **5:1 cost matrix** |
| Fixed 0.5 threshold | **Threshold tuning** inside the training folds (cost for German, MCC for Taiwan) |
| Single 10-fold run, no variance or significance tests | Repeated stratified CV and a **corrected resampled t-test** (Nadeau and Bengio) |
| Best-of-30 selection bias | A **primary pipeline declared in advance**; others reported as secondary |
| No fairness audit | Group-wise selection rate, TPR and FPR on out-of-fold predictions |
| Possible leakage inflating results | Leakage experiment: resample-before-CV vs in-fold |
| Poor calibration not checked | Brier score and reliability curves, raw vs sigmoid-calibrated |

**Order of work:** run German first (fast), then Taiwan (slow). Every finished configuration is cached, so a disconnect does not lose results.

## Stage 0 – Setup and configuration

In [ ]:
%pip -q install imbalanced-learn xlrd openpyxl xgboost

In [ ]:
import os, sys, glob, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import loguniform, uniform, randint

import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import (StratifiedKFold, RepeatedStratifiedKFold, RandomizedSearchCV,
                                     cross_validate, cross_val_predict)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (make_scorer, confusion_matrix, f1_score, recall_score, precision_score,
                             matthews_corrcoef, brier_score_loss)
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC
from imblearn.combine import SMOTEENN
try:
    from sklearn.model_selection import TunedThresholdClassifierCV
    HAS_THR = True
except ImportError:
    HAS_THR = False
    print("WARNING: scikit-learn >= 1.5 is needed for threshold tuning; upgrade with: %pip install -U scikit-learn")

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
# ------------------------- CONFIGURATION (edit here only) -------------------------
SEED = 42
QUICK_TEST = True        # True = tiny dry run (few folds, few iterations, small Taiwan subsample)

DATASETS = ["German", "Taiwan"]                  # run ["German"] first
MODELS_USED = {"German": ["LogReg", "RandomForest", "XGBoost", "MLP"],
               "Taiwan": ["LogReg", "RandomForest", "XGBoost", "MLP"]}
VARIANTS = ["paper_style", "fixed_none", "fixed_smotenc", "fixed_smoteenn"]

OUTER_FOLDS = 10                                  # Stage A: repeated stratified CV
REPEATS = {"German": 3, "Taiwan": 1}
OUTER_TUNED = {"German": 10, "Taiwan": 5}         # Stage B: nested CV outer folds (Taiwan reduced for run time)
INNER_FOLDS = 3                                   # hyperparameter search folds
THR_FOLDS = 3                                     # threshold-tuning folds
N_ITER = {"German": 20, "Taiwan": 6}              # random-search candidates

# Primary pipeline DECLARED IN ADVANCE (avoids best-of-many selection bias); others are secondary
PRIMARY = {"German": ("XGBoost", "fixed_none"), "Taiwan": ("XGBoost", "fixed_none")}
CANDIDATES = {"German": [("XGBoost", "fixed_none"), ("RandomForest", "fixed_smotenc"), ("MLP", "fixed_smotenc")],
              "Taiwan": [("XGBoost", "fixed_none"), ("RandomForest", "fixed_smotenc")]}
THR_OBJECTIVE = {"German": "cost", "Taiwan": "mcc"}   # decision-threshold objective
COST_FN, COST_FP = 5, 1                           # German documentation: bad->good costs 5, good->bad costs 1
TOL = 0.02                                        # 'comparable' band for the verdict
QUICK_ROWS = 3000                                 # Taiwan subsample size in QUICK_TEST

if QUICK_TEST:
    OUTER_FOLDS = 3; REPEATS = {"German": 1, "Taiwan": 1}; OUTER_TUNED = {"German": 3, "Taiwan": 3}
    INNER_FOLDS = 2; THR_FOLDS = 2; N_ITER = {"German": 2, "Taiwan": 2}

np.random.seed(SEED)
IN_COLAB = "google.colab" in sys.modules

def find_data_dir():
    # Works in Colab and in a local project (Antigravity): looks for german.data in the usual places
    for c in ["data/raw", "../data/raw", "data", "../data", "/content/data"]:
        if os.path.exists(os.path.join(c, "german.data")):
            return os.path.abspath(c)
    return os.path.abspath("/content/data" if IN_COLAB else "data/raw")

DATA_DIR = find_data_dir()
RESULTS_DIR = (os.path.abspath(os.path.join(DATA_DIR, "..", "..", "results"))
               if DATA_DIR.endswith(os.path.join("data", "raw")) else os.path.join(os.path.dirname(DATA_DIR), "results"))
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)
print("data dir:", DATA_DIR, "| results dir:", RESULTS_DIR)

## Stage 1 – Load, clean and document the datasets
Download the **originals from UCI** (cite UCI, not Kaggle mirrors):
- `german.data` (Statlog German Credit)
- `default of credit card clients.xls` (Taiwan)

Run the next cell and upload both files when prompted (or place them in `/content/data`).

In [ ]:
g_ok = os.path.exists(f"{DATA_DIR}/german.data")
t_ok = len(glob.glob(f"{DATA_DIR}/*default*credit*")) > 0
if not (g_ok and t_ok) and IN_COLAB:
    from google.colab import files
    print("Upload german.data and the Taiwan file (.xls / .xlsx / .csv)")
    for name, content in files.upload().items():
        open(f"{DATA_DIR}/{name}", "wb").write(content)
print(os.listdir(DATA_DIR))

In [ ]:
CLEAN_LOG = []

GERMAN_COLS = ["checking_status", "duration", "credit_history", "purpose", "credit_amount", "savings",
               "employment", "installment_rate", "personal_status", "other_debtors", "residence_since",
               "property", "age", "other_installment_plans", "housing", "existing_credits", "job",
               "num_dependents", "telephone", "foreign_worker", "target"]

def load_german():
    df = pd.read_csv(f"{DATA_DIR}/german.data", sep=r"\s+", header=None, names=GERMAN_COLS)
    y = (df.pop("target") == 2).astype(int)                # 2 = bad credit -> positive (minority) class
    CLEAN_LOG.append(("German", "missing values", int(df.isna().sum().sum())))
    CLEAN_LOG.append(("German", "duplicate rows removed", int(df.duplicated().sum())))
    keep = ~df.duplicated()
    df, y = df[keep].reset_index(drop=True), y[keep].reset_index(drop=True)
    cat = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    num = [c for c in df.columns if c not in cat]
    return df, y, cat, num

def load_taiwan():
    path = glob.glob(f"{DATA_DIR}/*default*credit*")[0]
    df = pd.read_csv(path) if path.lower().endswith(".csv") else pd.read_excel(path, header=1)
    df = df.drop(columns=[c for c in df.columns if str(c).strip().upper() == "ID"])
    target = [c for c in df.columns if "default" in str(c).lower()][0]
    y = df.pop(target).astype(int)                         # 1 = default -> positive (minority) class
    df = df.rename(columns={"PAY_0": "PAY_1"})             # fix naming inconsistency (PAY_0 -> PAY_1)
    CLEAN_LOG.append(("Taiwan", "missing values", int(df.isna().sum().sum())))
    for col, bad, to in (("EDUCATION", [0, 5, 6], 4), ("MARRIAGE", [0], 3)):
        n = int(df[col].isin(bad).sum())
        CLEAN_LOG.append(("Taiwan", f"{col} undocumented codes {bad} merged into 'others' ({to})", n))
        df[col] = df[col].replace({b: to for b in bad})
    dup = df.duplicated()
    y_dup = pd.concat([df, y], axis=1).duplicated()
    CLEAN_LOG.append(("Taiwan", "duplicate rows removed", int(y_dup.sum())))
    keep = ~y_dup
    df, y = df[keep].reset_index(drop=True), y[keep].reset_index(drop=True)
    cat = ["SEX", "EDUCATION", "MARRIAGE"]                 # same 3 categorical features as the paper
    for c in cat:
        df[c] = df[c].astype(str)
    num = [c for c in df.columns if c not in cat]
    if QUICK_TEST and len(df) > QUICK_ROWS:
        idx = df.sample(QUICK_ROWS, random_state=SEED).index
        df, y = df.loc[idx].reset_index(drop=True), y.loc[idx].reset_index(drop=True)
    return df, y, cat, num

LOADERS = {"German": load_german, "Taiwan": load_taiwan}
DATA = {ds: LOADERS[ds]() for ds in DATASETS}

print("Cleaning log:")
display(pd.DataFrame(CLEAN_LOG, columns=["dataset", "step", "rows/values affected"]))
rows = []
for ds, (X, y, cat, num) in DATA.items():
    rows.append(dict(dataset=ds, rows=len(X), features=X.shape[1], categorical=len(cat), numeric=len(num),
                     minority_pct=round(100 * y.mean(), 1)))
display(pd.DataFrame(rows).set_index("dataset"))
PRIOR_ODDS = {ds: float((1 - DATA[ds][1].mean()) / DATA[ds][1].mean()) for ds in DATA}   # for XGBoost scale_pos_weight

## Stage 2 – Pipelines: the paper's as described vs corrected
| Variant | Encoding | Scaling | Resampling | Purpose |
|---|---|---|---|---|
| `paper_style` | ordinal codes fed to the model | none | SMOTE-ENN on the encoded data | The paper's pipeline under its **unstated** choices |
| `fixed_none` | one-hot | signed-log + standardise (in-fold) | none; **class weights** where supported | Cost-sensitive learning, no synthetic data |
| `fixed_smotenc` | one-hot (after SMOTE-NC) | signed-log + standardise (in-fold) | **SMOTE-NC** (categorical-aware) | Fixes SMOTE on categorical features |
| `fixed_smoteenn` | one-hot | signed-log + standardise (in-fold) | SMOTE-ENN | Isolates the effect of *preprocessing* from *resampling* |

All preprocessing and resampling sit inside the pipeline, so they are refit on each training fold only.

In [ ]:
class SkewLog(BaseEstimator, TransformerMixin):
    # Signed log1p on features whose |skewness| exceeds a threshold; skewness is learned on the training data only.
    def __init__(self, threshold=1.0):
        self.threshold = threshold
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        sk = pd.DataFrame(X).skew().fillna(0).values
        self.cols_ = np.where(np.abs(sk) > self.threshold)[0]
        return self
    def transform(self, X):
        X = np.array(X, dtype=float, copy=True)
        for c in self.cols_:
            X[:, c] = np.sign(X[:, c]) * np.log1p(np.abs(X[:, c]))
        return X

def num_block():
    from sklearn.pipeline import Pipeline
    return Pipeline([("log", SkewLog()), ("sc", StandardScaler())])

def pre_fixed(cat, num):
    return ColumnTransformer([("num", num_block(), num),
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)],
                             sparse_threshold=0)

def pre_ordinal(cat, num, scale):
    return ColumnTransformer([("num", num_block() if scale else "passthrough", num),
                              ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat)],
                             sparse_threshold=0)

def make_model(name, variant, ds):
    weighted = variant == "fixed_none"
    if name == "LogReg":
        return LogisticRegression(max_iter=2000, class_weight="balanced" if weighted else None, random_state=SEED)
    if name == "RandomForest":
        return RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample" if weighted else None,
                                      random_state=SEED, n_jobs=1)
    if name == "XGBoost":
        return XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                             scale_pos_weight=PRIOR_ODDS[ds] if weighted else 1.0, eval_metric="logloss",
                             tree_method="hist", random_state=SEED, n_jobs=1, verbosity=0)
    if name == "MLP":
        return MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=300, random_state=SEED)
    raise ValueError(name)

def make_pipe(ds, model_name, variant):
    _, _, cat, num = DATA[ds]
    clf = make_model(model_name, variant, ds)
    n_num, n_cat = len(num), len(cat)
    if variant == "paper_style":
        steps = [("pre", pre_ordinal(cat, num, scale=False)), ("res", SMOTEENN(random_state=SEED)), ("clf", clf)]
    elif variant == "fixed_none":
        steps = [("pre", pre_fixed(cat, num)), ("clf", clf)]
    elif variant == "fixed_smoteenn":
        steps = [("pre", pre_fixed(cat, num)), ("res", SMOTEENN(random_state=SEED)), ("clf", clf)]
    elif variant == "fixed_smotenc":
        cat_idx = list(range(n_num, n_num + n_cat))
        steps = [("pre1", pre_ordinal(cat, num, scale=True)),
                 ("res", SMOTENC(categorical_features=cat_idx, random_state=SEED)),
                 ("pre2", ColumnTransformer([("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_idx)],
                                            remainder="passthrough", sparse_threshold=0)),
                 ("clf", clf)]
    else:
        raise ValueError(variant)
    return ImbPipeline(steps)

PARAMS = {
    "LogReg": {"clf__C": loguniform(1e-3, 1e2)},
    "RandomForest": {"clf__n_estimators": randint(100, 400), "clf__max_depth": [None, 6, 10, 16],
                     "clf__min_samples_leaf": randint(1, 10), "clf__max_features": ["sqrt", "log2", 0.5]},
    "XGBoost": {"clf__n_estimators": randint(100, 400), "clf__max_depth": randint(2, 7),
                "clf__learning_rate": loguniform(0.01, 0.3), "clf__subsample": uniform(0.6, 0.4),
                "clf__colsample_bytree": uniform(0.6, 0.4), "clf__min_child_weight": randint(1, 8)},
    "MLP": {"clf__hidden_layer_sizes": [(32,), (64,), (64, 32), (128, 64)], "clf__alpha": loguniform(1e-5, 1e-1),
            "clf__learning_rate_init": loguniform(1e-4, 1e-2)},
}
print("pipelines ready:", VARIANTS)

## Stage 3 – Metrics
The paper reports accuracy, recall, specificity, precision and F1. Added here: **MCC**, **balanced accuracy**, **ROC-AUC**, **PR-AUC**, **Brier score**, and for German the **expected misclassification cost per customer** (bad→good costs 5, good→bad costs 1).

In [ ]:
def specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fp)

def misclass_cost(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return (COST_FN * fn + COST_FP * fp) / len(y_true)

def get_scoring(ds):
    s = {"accuracy": "accuracy", "balanced_accuracy": "balanced_accuracy", "recall": "recall",
         "specificity": make_scorer(specificity), "precision": make_scorer(precision_score, zero_division=0),
         "f1": "f1", "f1_macro": "f1_macro", "f1_weighted": "f1_weighted",
         "mcc": make_scorer(matthews_corrcoef), "roc_auc": "roc_auc", "pr_auc": "average_precision",
         "brier": "neg_brier_score"}
    if ds == "German":
        s["cost"] = make_scorer(misclass_cost, greater_is_better=False)
    return s

def thr_scorer(obj):
    return {"cost": make_scorer(misclass_cost, greater_is_better=False),
            "mcc": make_scorer(matthews_corrcoef), "f1": make_scorer(f1_score)}[obj]

def best_threshold(y, p, obj):
    grid = np.linspace(0.02, 0.98, 97)
    if obj == "cost":
        vals = [-misclass_cost(y, (p >= t).astype(int)) for t in grid]
    elif obj == "mcc":
        vals = [matthews_corrcoef(y, (p >= t).astype(int)) for t in grid]
    else:
        vals = [f1_score(y, (p >= t).astype(int)) for t in grid]
    return float(grid[int(np.argmax(vals))])

## Stage 4 – Experiment engine
* **Default mode (Stage A):** default hyperparameters, threshold 0.5, repeated stratified 10-fold CV. Compares the pipeline variants fairly.
* **Tuned mode (Stage B):** nested CV. Inner loop: random search for hyperparameters (scored by PR-AUC, which is threshold-free). Threshold tuned on top inside the training fold (cost for German, MCC for Taiwan). The outer folds never see tuning.

Results are cached as JSON, with per-fold scores kept for paired significance tests.

In [ ]:
SIGN = {"brier": -1, "cost": -1}     # scorers that return negated values

def cache_path(ds, model, variant, mode, obj):
    return f"{RESULTS_DIR}/{ds}__{model}__{variant}__{mode}__{obj}__q{int(QUICK_TEST)}.json"

def get_cv(ds, tuned):
    if tuned:
        return RepeatedStratifiedKFold(n_splits=OUTER_TUNED[ds], n_repeats=1, random_state=SEED)
    return RepeatedStratifiedKFold(n_splits=OUTER_FOLDS, n_repeats=REPEATS[ds], random_state=SEED)

def build_estimator(ds, model, variant, mode, obj):
    est = make_pipe(ds, model, variant)
    if mode == "tuned_thr":
        inner = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=SEED)
        est = RandomizedSearchCV(est, PARAMS[model], n_iter=N_ITER[ds], scoring="average_precision",
                                 cv=inner, random_state=SEED, n_jobs=-1, refit=True)
        if HAS_THR:
            est = TunedThresholdClassifierCV(est, scoring=thr_scorer(obj), cv=THR_FOLDS, refit=True,
                                             random_state=SEED)
    return est

def run_experiment(ds, model, variant, mode="default", obj=None):
    path = cache_path(ds, model, variant, mode, obj)
    if os.path.exists(path):
        return
    X, y, _, _ = DATA[ds]
    est = build_estimator(ds, model, variant, mode, obj)
    tuned = mode == "tuned_thr"
    t0 = time.time()
    res = cross_validate(est, X, y, cv=get_cv(ds, tuned), scoring=get_scoring(ds),
                         n_jobs=1 if tuned else -1, return_estimator=tuned)
    out = dict(dataset=ds, model=model, variant=variant, mode=mode, objective=obj, folds={})
    for m in get_scoring(ds):
        v = SIGN.get(m, 1) * np.asarray(res[f"test_{m}"])
        out[f"{m}_mean"], out[f"{m}_std"] = float(v.mean()), float(v.std(ddof=1))
        out["folds"][m] = [float(a) for a in v]
    if tuned:
        bp, thr = [], []
        for e in res["estimator"]:
            inner = e.estimator_ if hasattr(e, "best_threshold_") else e
            bp.append(getattr(inner, "best_params_", {}))
            if hasattr(e, "best_threshold_"):
                thr.append(float(e.best_threshold_))
        out["best_params"], out["thresholds"] = bp, thr
    out["seconds"] = round(time.time() - t0, 1)
    with open(path, "w") as f:
        json.dump(out, f, default=str)
    print(f"[{ds}] {model:12s} {variant:14s} {mode:9s} {str(obj):5s} MCC={out['mcc_mean']:.3f}  "
          f"F1={out['f1_mean']:.3f}  PR-AUC={out['pr_auc_mean']:.3f}  ({out['seconds']}s)")

def load_all():
    R = {}
    for p in glob.glob(f"{RESULTS_DIR}/*__q{int(QUICK_TEST)}.json"):
        r = json.load(open(p))
        R[(r["dataset"], r["model"], r["variant"], r["mode"], r["objective"])] = r
    return R

def table(R, ds, mode, obj=None, cols=("recall", "specificity", "precision", "f1", "mcc", "pr_auc", "roc_auc", "brier")):
    rows = []
    for (d_, m, v, md_, ob), r in R.items():
        if d_ == ds and md_ == mode and (obj is None or ob == obj):
            row = {"model": m, "variant": v, "objective": ob}
            for c in list(cols) + (["cost"] if ds == "German" else []):
                row[c] = f"{r[c + '_mean']:.3f} ± {r[c + '_std']:.3f}"
            row["_mcc"] = r["mcc_mean"]
            rows.append(row)
    return pd.DataFrame(rows).sort_values("_mcc", ascending=False).drop(columns="_mcc").reset_index(drop=True)

## Stage A – Compare the four pipeline variants (default hyperparameters)
This isolates the **effect of the fixes**: same models, same folds, only the preprocessing and resampling differ.

In [ ]:
for ds in DATASETS:
    for m in MODELS_USED[ds]:
        for v in VARIANTS:
            run_experiment(ds, m, v, "default", None)
R = load_all()
for ds in DATASETS:
    print(f"\n===== {ds} Credit – Stage A (mean ± SD over {OUTER_FOLDS * REPEATS[ds]} folds) =====")
    display(table(R, ds, "default"))

In [ ]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(6.5 * len(DATASETS), 4.2), squeeze=False)
for ax, ds in zip(axes[0], DATASETS):
    rows = [(m, v, R[(ds, m, v, "default", None)]["mcc_mean"]) for m in MODELS_USED[ds] for v in VARIANTS
            if (ds, m, v, "default", None) in R]
    pv = pd.DataFrame(rows, columns=["model", "variant", "mcc"]).pivot(index="model", columns="variant", values="mcc")
    pv[[v for v in VARIANTS if v in pv.columns]].plot(kind="bar", ax=ax, rot=0)
    ax.set_title(f"{ds}: MCC by model and pipeline variant"); ax.set_ylabel("MCC"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

### Do the fixes make a statistically meaningful difference?
Paired comparison on identical folds: each corrected variant vs `paper_style`, using the **corrected resampled t-test** (Nadeau and Bengio), which accounts for overlapping training sets. A positive difference means the corrected pipeline is better.

In [ ]:
def corrected_ttest(diffs, test_train_ratio):
    k = len(diffs)
    var = np.var(diffs, ddof=1)
    if var == 0 or k < 2:
        return np.nan, np.nan
    t = np.mean(diffs) / np.sqrt((1 / k + test_train_ratio) * var)
    return float(t), float(2 * stats.t.sf(abs(t), k - 1))

rows = []
ratio = 1 / (OUTER_FOLDS - 1)
for ds in DATASETS:
    for m in MODELS_USED[ds]:
        base = R.get((ds, m, "paper_style", "default", None))
        for v in ["fixed_none", "fixed_smotenc", "fixed_smoteenn"]:
            cur = R.get((ds, m, v, "default", None))
            if base is None or cur is None:
                continue
            for metric in ["mcc", "pr_auc", "f1"]:
                d = np.array(cur["folds"][metric]) - np.array(base["folds"][metric])
                t, p = corrected_ttest(d, ratio)
                rows.append(dict(dataset=ds, model=m, variant=v, metric=metric, paper_style=round(np.mean(base["folds"][metric]), 3),
                                 corrected=round(np.mean(cur["folds"][metric]), 3), diff=round(d.mean(), 3), p_value=round(p, 4)))
cmp_df = pd.DataFrame(rows)
display(cmp_df[cmp_df.metric == "mcc"].reset_index(drop=True))

## Stage B – Nested-CV tuning with threshold optimisation (candidates)
**Primary pipelines were declared in advance** in the config (`PRIMARY`), so the headline result is not the best of many looks. The other candidates are secondary. Taiwan uses 5 outer folds to keep the run time realistic.

> **Run time:** German finishes in minutes. Taiwan can take an hour or more; start with fewer candidates or a smaller `N_ITER` if needed. Results are cached.

In [ ]:
for ds in DATASETS:
    for (m, v) in CANDIDATES[ds]:
        run_experiment(ds, m, v, "tuned_thr", THR_OBJECTIVE[ds])
    pm, pv = PRIMARY[ds]
    run_experiment(ds, pm, pv, "tuned_thr", "f1")     # F1-optimal threshold: only for comparison with the authors' F1
R = load_all()
for ds in DATASETS:
    print(f"\n===== {ds} Credit – tuned, threshold objective = {THR_OBJECTIVE[ds]} =====")
    display(table(R, ds, "tuned_thr", THR_OBJECTIVE[ds]))
    pm, pv = PRIMARY[ds]
    r = R[(ds, pm, pv, "tuned_thr", THR_OBJECTIVE[ds])]
    print("Primary pipeline thresholds chosen per outer fold:", np.round(r["thresholds"], 2).tolist() if r.get("thresholds") else "n/a")

## Stage C – Verdict against the authors
Verdict rule, fixed before looking at the numbers: for each metric, `delta = mine − authors`, `band = max(TOL, my fold SD)`.
`delta > band` → **Mine better**, `delta < −band` → **Authors better**, otherwise **Comparable**.

**Fill in the authors' other metrics** (accuracy, recall, specificity, precision) from the paper's tables if you have them. The two F1 values are the headline results recorded in Phase 1.

Caveat: the authors' numbers come from *their* folds, so no paired test is possible. Their F1 definition is not stated, so my binary, macro and weighted F1 are all shown.

In [ ]:
AUTHORS = {
    "German": {"model": "MLP + SMOTE-ENN",           "f1": 0.937, "accuracy": None, "recall": None, "specificity": None, "precision": None},
    "Taiwan": {"model": "Random Forest + SMOTE-ENN", "f1": 0.610, "accuracy": None, "recall": None, "specificity": None, "precision": None},
}
rows, replic = [], []
for ds in DATASETS:
    pm, pv = PRIMARY[ds]
    r = R.get((ds, pm, pv, "tuned_thr", "f1"))
    if r is None:
        print(f"[{ds}] run Stage B first"); continue
    a = AUTHORS[ds]
    print(f"\n===== {ds} Credit =====")
    print(f"My primary pipeline: {pm} + {pv}, tuned (nested CV), F1-optimal threshold   |   Authors' best: {a['model']}")
    print("My F1 variants -> binary: {:.3f} | macro: {:.3f} | weighted: {:.3f}   (authors' F1 = {})".format(
        r["f1_mean"], r["f1_macro_mean"], r["f1_weighted_mean"], a["f1"]))
    for m in ["f1", "recall", "precision", "specificity", "accuracy"]:
        if a.get(m) is None:
            continue
        mine, sd = r[f"{m}_mean"], r[f"{m}_std"]
        delta, band = mine - a[m], max(TOL, sd)
        v = "Mine better" if delta > band else ("Authors better" if delta < -band else "Comparable")
        rows.append(dict(dataset=ds, metric=m, mine=round(mine, 3), authors=a[m], delta=round(delta, 3), band=round(band, 3), verdict=v))
    # my replication of the authors' best model type under the paper-style pipeline
    am = "MLP" if ds == "German" else "RandomForest"
    rp = R.get((ds, am, "paper_style", "default", None))
    if rp:
        replic.append(dict(dataset=ds, pipeline=f"{am} paper_style (default hyperparameters)", f1=round(rp["f1_mean"], 3),
                           f1_macro=round(rp["f1_macro_mean"], 3), f1_weighted=round(rp["f1_weighted_mean"], 3),
                           mcc=round(rp["mcc_mean"], 3), authors_f1=a["f1"]))
if replic:
    print("\nMy replication of the authors' best model type (paper-style pipeline):")
    display(pd.DataFrame(replic))
if rows:
    vt = pd.DataFrame(rows); display(vt)
    for ds in vt.dataset.unique():
        c = vt[vt.dataset == ds].verdict.value_counts()
        mine, auth = c.get("Mine better", 0), c.get("Authors better", 0)
        overall = "MINE is better" if mine > auth else ("AUTHORS are better" if auth > mine else "COMPARABLE overall")
        print(f"[{ds}] Mine better: {mine} | Authors better: {auth} | Comparable: {c.get('Comparable', 0)}  ->  {overall}")

### Reading the verdict honestly
* If the authors' F1 is far above everything achievable in-fold (German 0.937), do not conclude that you did something wrong: check the **leakage experiment** below and the F1 variant (weighted/macro F1 are always higher than minority-class F1).
* A **tie is a valid finding**: it questions the benefit of deep learning for small tabular data.
* The primary pipeline was fixed in advance, but the threshold objective and tolerance band are still choices you must state in the report.

## Stage D – Could leakage explain very high published scores?
Resampling the **whole dataset before CV** (a common error) vs resampling **inside** each training fold. The leaky score is computed on the resampled data, as it would be if the pipeline were wrong. This does not prove what the authors did; it shows how much a leaky pipeline can inflate results.

In [ ]:
def leakage_experiment(ds, model="RandomForest"):
    X, y, cat, num = DATA[ds]
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    sc = {"f1": "f1", "recall": "recall", "precision": make_scorer(precision_score, zero_division=0),
          "mcc": make_scorer(matthews_corrcoef)}
    Z = pre_ordinal(cat, num, scale=False).fit_transform(X)
    Zr, yr = SMOTEENN(random_state=SEED).fit_resample(Z, y)
    leaky = cross_validate(make_model(model, "paper_style", ds), Zr, yr, cv=cv, scoring=sc, n_jobs=-1)
    proper = cross_validate(make_pipe(ds, model, "paper_style"), X, y, cv=cv, scoring=sc, n_jobs=-1)
    row = {"dataset": ds, "model": model}
    for k in sc:
        row[f"leaky_{k}"] = round(leaky[f"test_{k}"].mean(), 3)
        row[f"in-fold_{k}"] = round(proper[f"test_{k}"].mean(), 3)
    row["F1 inflation"] = round(row["leaky_f1"] - row["in-fold_f1"], 3)
    return row
leak = pd.DataFrame([leakage_experiment(ds) for ds in DATASETS])
display(leak)

## Stage E – Calibration and fairness audit (primary pipeline)
Out-of-fold predictions (5-fold) of the primary pipeline with default hyperparameters. The decision threshold is chosen on these predictions for the audit only (cost for German, MCC for Taiwan).

* **Calibration:** Brier score and reliability curves, raw vs sigmoid-calibrated. Class weighting inflates predicted probabilities, so raw probabilities are not real default risks.
* **Fairness:** selection rate, true-positive rate and false-positive rate per group. The paper keeps sensitive attributes as features and does no audit; this is a **diagnostic, not a mitigation**.

In [ ]:
def oof_probs(ds, calibrated):
    X, y, _, _ = DATA[ds]
    pm, pv = PRIMARY[ds]
    est = make_pipe(ds, pm, pv)
    if calibrated:
        est = CalibratedClassifierCV(est, method="sigmoid", cv=3)
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    return cross_val_predict(est, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

fig, axes = plt.subplots(1, len(DATASETS), figsize=(5.5 * len(DATASETS), 4.2), squeeze=False)
OOF, cal_rows = {}, []
for ax, ds in zip(axes[0], DATASETS):
    y = DATA[ds][1].values
    p_raw, p_cal = oof_probs(ds, False), oof_probs(ds, True)
    OOF[ds] = p_raw
    for name, p in (("raw", p_raw), ("sigmoid-calibrated", p_cal)):
        fr, mp = calibration_curve(y, p, n_bins=8, strategy="quantile")
        ax.plot(mp, fr, marker="o", label=f"{name} (Brier {brier_score_loss(y, p):.3f})")
        cal_rows.append(dict(dataset=ds, probabilities=name, brier=round(brier_score_loss(y, p), 4)))
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_title(f"{ds}: reliability curve ({PRIMARY[ds][0]})"); ax.set_xlabel("predicted probability"); ax.set_ylabel("observed default rate")
    ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
display(pd.DataFrame(cal_rows))

In [ ]:
def group_report(y, pred, g, name):
    rows = []
    for val in sorted(pd.unique(g)):
        m = (g == val)
        yt, yp = y[m], pred[m]
        rows.append(dict(attribute=name, group=val, n=int(m.sum()), base_rate=round(yt.mean(), 3), selection_rate=round(yp.mean(), 3),
                         TPR=round(((yp == 1) & (yt == 1)).sum() / max((yt == 1).sum(), 1), 3),
                         FPR=round(((yp == 1) & (yt == 0)).sum() / max((yt == 0).sum(), 1), 3)))
    return pd.DataFrame(rows)

def group_defs(ds):
    X = DATA[ds][0]
    if ds == "German":
        return {"sex": np.where(X.personal_status == "A92", "female", "male"),
                "age": np.where(X.age < 25, "under 25", "25 and over"),
                "foreign_worker": np.where(X.foreign_worker == "A201", "foreign worker", "not foreign")}
    return {"sex": np.where(X.SEX == "2", "female", "male"),
            "marriage": X.MARRIAGE.map({"1": "married", "2": "single", "3": "other"}).values,
            "age": pd.cut(X.AGE, [0, 29, 45, 120], labels=["under 30", "30-45", "over 45"]).astype(str).values}

audit = []
for ds in DATASETS:
    y = DATA[ds][1].values
    thr = best_threshold(y, OOF[ds], THR_OBJECTIVE[ds])
    pred = (OOF[ds] >= thr).astype(int)
    print(f"\n===== {ds}: audit threshold = {thr:.2f} (objective {THR_OBJECTIVE[ds]}) =====")
    for name, g in group_defs(ds).items():
        rep = group_report(y, pred, g, name)
        display(rep)
        audit.append(dict(dataset=ds, attribute=name,
                          demographic_parity_gap=round(rep.selection_rate.max() - rep.selection_rate.min(), 3),
                          equal_opportunity_gap=round(rep.TPR.max() - rep.TPR.min(), 3),
                          FPR_gap=round(rep.FPR.max() - rep.FPR.min(), 3)))
print("\nGap summary (max - min across groups):")
display(pd.DataFrame(audit))

## Stage F – Noise stress test (hypothesis for Phase 3d)
Gaussian noise (scaled to each numeric feature's training SD) is added to the **validation features only**. **Write your hypothesis before running:** linear and distance-based pipelines degrade faster than tree ensembles as noise grows.

In [ ]:
def noise_test(ds, model_name, variant="fixed_none", levels=(0, 0.1, 0.25, 0.5, 1.0)):
    X, y, cat, num = DATA[ds]
    rng = np.random.default_rng(SEED)
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    rows = []
    for tr, te in cv.split(X, y):
        pipe = make_pipe(ds, model_name, variant).fit(X.iloc[tr], y.iloc[tr])
        sd = X.iloc[tr][num].astype(float).std().values
        for lvl in levels:
            Xn = X.iloc[te].copy()
            Xn[num] = Xn[num].astype(float).values + rng.normal(0, 1, (len(Xn), len(num))) * sd * lvl
            p = pipe.predict(Xn)
            rows.append(dict(noise_sd=lvl, mcc=matthews_corrcoef(y.iloc[te], p), recall=recall_score(y.iloc[te], p),
                             f1=f1_score(y.iloc[te], p)))
    out = pd.DataFrame(rows).groupby("noise_sd").mean()
    out.insert(0, "model", model_name); out.insert(0, "dataset", ds)
    return out.reset_index()

noise = pd.concat([noise_test(ds, m) for ds in DATASETS for m in MODELS_USED[ds]], ignore_index=True)
fig, axes = plt.subplots(1, len(DATASETS), figsize=(6 * len(DATASETS), 4), squeeze=False)
for ax, ds in zip(axes[0], DATASETS):
    for m in MODELS_USED[ds]:
        s = noise[(noise.dataset == ds) & (noise.model == m)]
        ax.plot(s.noise_sd, s.mcc, marker="o", label=m)
    ax.set_title(f"{ds}: MCC vs noise level"); ax.set_xlabel("noise (x feature SD)"); ax.set_ylabel("MCC"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
display(noise.round(3))

## Stage G – Export tables for the report

In [ ]:
R = load_all()
cols = ["dataset", "model", "variant", "mode", "objective"] + [f"{m}_{s}" for m in get_scoring("Taiwan") for s in ("mean", "std")]
pd.DataFrame([{k: r.get(k) for k in cols} for r in R.values()]).to_csv(f"{RESULTS_DIR}/all_results.csv", index=False)
cmp_df.to_csv(f"{RESULTS_DIR}/fix_vs_paper_style_tests.csv", index=False)
leak.to_csv(f"{RESULTS_DIR}/leakage_experiment.csv", index=False)
noise.to_csv(f"{RESULTS_DIR}/noise_stress_test.csv", index=False)
pd.DataFrame(audit).to_csv(f"{RESULTS_DIR}/fairness_gaps.csv", index=False)
pd.DataFrame(cal_rows).to_csv(f"{RESULTS_DIR}/calibration.csv", index=False)
if rows:
    pd.DataFrame(rows).to_csv(f"{RESULTS_DIR}/verdicts.csv", index=False)
print("Saved:", sorted(os.listdir(RESULTS_DIR))[:12], "...")

## Conclusions (complete after running, using the printed numbers)
1. **Effect of the fixes (Stage A):** which corrected variant beat `paper_style`, by how much (MCC), and was it significant (corrected t-test p-values)?
2. **German:** primary pipeline F1 = ___ vs authors' ___ → verdict ___. Expected cost per customer = ___.
3. **Taiwan:** primary pipeline F1 = ___ vs authors' ___ → verdict ___.
4. **Leakage experiment:** F1 inflation = ___ (German), ___ (Taiwan). What does this suggest about very high published scores?
5. **Calibration:** Brier raw vs calibrated = ___; are raw probabilities usable as risk estimates?
6. **Fairness:** largest gaps (parity / TPR / FPR) = ___; which groups are affected?
7. **Noise test:** most robust model = ___; supports or contradicts the hypothesis because ___.
8. **Limitations:** authors' numbers come from different folds; MLP here is a simple scikit-learn network, not the authors' architecture; Taiwan tuning used 5 outer folds and few search iterations.